# Chapter 1: Introduction and Overview
*From: Database Internals*

## TL;DR

This chapter is a **taxonomy cell** for the rest of the book. It draws the distinctions that every later chapter assumes:

- A DBMS is a **layered system**: transport -> query processor -> optimizer -> execution engine -> storage engine (which itself has transaction, lock, access-method, buffer, and recovery subcomponents).
- Databases are classified along **two primary storage axes**: *where the data lives* (memory vs disk) and *how it is laid out* (row- vs column-oriented, plus the hybrid "wide column" family).
- **Data files** hold records (heap, hashed, or index-organized), **index files** map keys to locations, and indexes are primary/secondary and clustered/non-clustered.
- Secondary indexes can reference records by **direct file offset** (fast reads, expensive relocations) or through the **primary-key index** (slower reads, cheaper updates) — a recurring design trade-off.
- Every storage structure in the book can be placed on three orthogonal axes: **buffering, immutability, and ordering**. B-Trees, LSM Trees, Bitcask, Bw-Trees, and WiscKey differ precisely in which corner of this cube they occupy.
- OLTP vs OLAP vs HTAP is orthogonal to all of the above and influences which combination of choices is appropriate.

## Chapter Roadmap

| # | Concept | One-line summary |
|---|---------|------------------|
| 1 | DBMS Architecture | The component stack a query traverses from wire to disk. |
| 2 | Memory vs Disk-Based Storage | Whether RAM or disk holds the *primary* copy of data, and what that forces on data structures and durability. |
| 3 | Row- vs Column-Oriented Layouts | Horizontal vs vertical partitioning of a table, plus wide-column families. |
| 4 | Data Files and Index Files | Heap/hash/index-organized tables, primary vs secondary indexes, and direct-offset vs primary-key indirection. |
| 5 | Buffering, Immutability, Ordering | The three axes that explain most storage-engine design differences. |

---

## 1. DBMS Architecture

A DBMS is organised around a **client/server model**: database nodes are the servers and application instances are the clients. There is no single canonical architecture, but most systems share the same component skeleton, differing mostly in where they draw the boundaries and how much work each layer does.

A request's journey starts at the **transport subsystem**, which handles both client traffic and inter-node cluster communication. From there it moves to the **query processor**, which parses, interprets, validates, and access-checks the query. The **query optimizer** then eliminates impossible and redundant pieces, estimates costs using index cardinality and data-placement statistics, and picks an execution plan. The plan itself is a dependency tree of operations that the **execution engine** drives, coordinating local work and any remote reads/writes/replication with other nodes.

Local operations bottom out in the **storage engine**, which is itself a cluster of focused managers: a *transaction manager* schedules and keeps the database logically consistent; a *lock manager* protects physical integrity under concurrency; *access methods* (heap files, B-Trees, LSM Trees) own on-disk organisation; a *buffer manager* caches pages in memory; and a *recovery manager* owns the write-ahead log and crash recovery. The transaction and lock managers together implement **concurrency control**.

```mermaid
graph TB
    Client[Client / Application]
    Client --> Transport[Transport Subsystem]
    Transport --> QP[Query Processor<br/>parse / validate / access control]
    QP --> Opt[Query Optimizer<br/>stats + cost -> execution plan]
    Opt --> Exec[Execution Engine<br/>local + remote ops, replication]
    Exec --> SE

    subgraph SE[Storage Engine]
        direction TB
        TM[Transaction Manager<br/>scheduling, consistency]
        LM[Lock Manager<br/>physical integrity]
        AM[Access Methods<br/>heap files, B-Trees, LSM Trees]
        BM[Buffer Manager<br/>page cache]
        RM[Recovery Manager<br/>WAL + crash recovery]
    end

    AM --> Disk[(On-disk data + index files)]
    BM --> Disk
    RM --> Disk
```

### Storage-engine subcomponents at a glance

| Component | Primary responsibility | Failure mode it prevents |
|-----------|------------------------|--------------------------|
| Transaction manager | Schedule transactions, maintain logical consistency | Partial / inconsistent state after multi-step ops |
| Lock manager | Guard database objects during concurrent ops | Physical data corruption from races |
| Access methods | Organise and retrieve records on disk | Inefficient scans / lookups |
| Buffer manager | Cache hot pages in RAM | Every op hitting disk |
| Recovery manager | Maintain WAL, replay on restart | Data loss after crash |

---

## 2. Memory- Versus Disk-Based DBMS

The primary storage medium fundamentally changes the design. **In-memory DBMSs** keep the working data almost entirely in RAM, using disk only for the write-ahead log and periodic backup snapshots. **Disk-based DBMSs** keep most data on disk and treat memory as a cache or scratch space. Accessing memory is several orders of magnitude faster than disk, which makes RAM-primary designs attractive, but RAM is volatile and still significantly more expensive per byte than SSDs or HDDs.

The difference is not just "same system, bigger page cache." An in-memory store can freely use pointer-chasing data structures (linked lists, skip lists, tries) because random memory access is cheap. A disk-based store must use **wide, short trees** (like B-Trees) to minimise the number of block I/Os per lookup, must pick a fixed serialization format, and must manage fragmentation and freed space manually. Handling variable-size values is "follow a pointer" in memory and "plan your page layout" on disk.

In-memory systems still need durability. They use a **write-ahead log** for every operation plus a periodic **backup copy** (snapshot) that is updated asynchronously in batches. Writing log records to the backup in batches produces a **checkpoint**, after which earlier log entries can be discarded — this bounds recovery time without blocking writers. Emerging **Non-Volatile Memory (NVM)** blurs the line further by offering byte-addressable, persistent storage with near-RAM latency.

```mermaid
graph LR
    Client -->|write| Log[Write-Ahead Log<br/>sequential, sync]
    Log -->|periodic batch apply| Snap[On-Disk Snapshot<br/>sorted backup]
    Snap -->|checkpoint boundary| Trunc[Discard older log entries]
    Crash((Crash)) -. replay .-> Recover[Snapshot + tail of log<br/>= recovered state]
    Snap -. restore .-> Recover
    Log -. replay tail .-> Recover
```

### Memory vs disk-based: head-to-head

| Dimension | In-Memory DBMS | Disk-Based DBMS |
|-----------|----------------|-----------------|
| Primary data location | RAM | Disk (SSD / HDD) |
| Disk role | WAL + periodic snapshot | Primary store; RAM is a cache |
| Durability mechanism | WAL + checkpointed snapshot | Page writes + WAL |
| Access granularity | Byte / pointer | Block / page |
| Typical data structures | Skip lists, tries, hash tables with pointer chasing | Wide/short B-Trees, LSM Trees, heap files |
| Variable-size data | Trivial (pointer) | Needs slotted pages, free-space management |
| Cost per byte | High (RAM) | Low (SSD/HDD) |
| Main limiters | RAM volatility + price | Seek latency, block I/O |

---

## 3. Row- Versus Column-Oriented Layouts

Tables are logically two-dimensional but disks are one-dimensional, so the system must **linearize** the table in some order. A **row-oriented** store writes whole rows contiguously — all fields of `user_id=10` sit together on the same page. A **column-oriented** store writes whole columns contiguously — every `price` value sits with other `price` values, possibly in a separate file per column. Since disks are accessed block-wise, this choice directly determines which queries read only the bytes they care about and which pay to load irrelevant fields.

Row-oriented stores (**MySQL, PostgreSQL**, and most traditional RDBMSs) match OLTP workloads where a request typically touches most fields of a record — reading a user profile, inserting an order row. Column-oriented stores (**MonetDB, C-Store/Vertica, Apache Parquet, ORC, Kudu, ClickHouse**) match OLAP workloads where queries aggregate a few columns across millions of rows. Column storage also enables **better compression** (same type and often similar values per column) and **vectorized (SIMD) execution**, amplifying the I/O savings with CPU savings. The cost is that reconstructing a full record requires reading and aligning every column, typically via an implicit positional **virtual ID**.

**Wide-column stores** (**Bigtable, HBase**) are a distinct third family. Conceptually the data is a multi-dimensional sorted map keyed by `(row key, column family, column qualifier, timestamp)`. Physically, column *families* are stored separately on disk, but **within** a family the data is laid out row-wise. The canonical example is Bigtable's Webtable: pages keyed by reversed URL, with `contents` and `anchor` column families stored as separate files.

```mermaid
graph TB
    subgraph Row[Row-Oriented: rows contiguous]
        R1[row 1: id name dob phone]
        R2[row 2: id name dob phone]
        R3[row 3: id name dob phone]
    end

    subgraph Col[Column-Oriented: columns contiguous]
        C1[col id: 1 2 3 4 ...]
        C2[col name: Ann Bob Cy ...]
        C3[col dob: ...]
        C4[col phone: ...]
    end

    subgraph Wide[Wide-Column: families separate, rows within family]
        F1[Family 'contents'<br/>rowkey -> html@t1, html@t2]
        F2[Family 'anchor'<br/>rowkey -> cnnsi.com, my.look.ca]
    end
```

### Layouts compared

| Aspect | Row-Oriented | Column-Oriented | Wide-Column |
|--------|--------------|-----------------|-------------|
| Contiguous unit | Whole row | Whole column | Row within a column family |
| Best for | OLTP: point reads/writes of full records | OLAP: aggregates, scans over few columns | Keyed access, sparse schemas, versioning |
| Compression potential | Moderate (mixed types in a page) | High (one type per column) | High within a family |
| Record reconstruction | Free | Needs virtual IDs / position alignment | Within a family: free |
| Typical write pattern | Whole-row inserts/updates | Bulk loads / append-heavy | Keyed upserts with timestamps |
| Examples | MySQL, PostgreSQL | MonetDB, C-Store, Parquet, ORC, Kudu, ClickHouse | Bigtable, HBase |

> **Deciding rule of thumb:** if queries mostly consume whole records and are point lookups or short range scans, pick row-oriented. If queries scan many rows but only a few columns — aggregates, trend analysis — pick column-oriented. Access pattern is the design input, not the schema.

---

## 4. Data Files and Index Files

A DBMS doesn't just dump records into filesystem directories; it composes its own files for three specific goals: **storage efficiency** (small overhead per record), **access efficiency** (few steps to locate a record), and **update efficiency** (few on-disk changes per write). Files are partitioned into **pages** (one or a few disk blocks each), which are themselves organised as record sequences or **slotted pages**. Most systems never delete in place — they write **tombstones** (deletion markers) and reclaim space later during garbage collection.

Records live in **data files**, which come in three organisational flavours. **Heap-organized tables** store records in write order with no logical order, so they're cheap to append but require a separate index to be searchable. **Hash-organized tables** bucket records by the hash of the key; within a bucket records may be append-ordered or sorted. **Index-organized tables (IOTs)** store data records *inside* the primary index itself — range scans become sequential traversals of the index, and lookups save at least one disk seek because there is no separate data file to hop to.

**Index files** map keys to locations. A **primary index** sits on the primary key (either explicit or an implicit auto-generated one); everything else is a **secondary index**. Secondary indexes may point either directly at the data record (a file offset, a.k.a. row locator) or at the primary key — and that second choice is the big design decision. Direct offsets make reads fast but make *every* record relocation (during updates, GC, or page splits) cascade into updates across *every* secondary index. Primary-key indirection (**MySQL InnoDB's approach**) pays an extra index lookup on every read, but makes relocations free for secondary indexes. A hybrid stores both and only falls back to the primary index when the cached offset turns out to be stale. Finally, an index whose order matches the data file's order is **clustered**; otherwise **non-clustered**. IOTs are clustered by construction; primary indexes are usually clustered; secondary indexes are non-clustered by definition.

```mermaid
graph TB
    subgraph Direct[Direct offset from secondary index]
        S1a[Secondary Index A] -->|offset| D1[(Data file / heap)]
        S2a[Secondary Index B] -->|offset| D1
    end

    subgraph Indirect[Indirection via primary key index]
        S1b[Secondary Index A] -->|PK| PK[Primary Index]
        S2b[Secondary Index B] -->|PK| PK
        PK -->|offset| D2[(Data file / IOT)]
    end
```

### Data file organisations

| Organisation | Record order | Needs separate primary index? | Strength | Weakness |
|--------------|--------------|-------------------------------|----------|----------|
| Heap-organized | Write order | Yes | Cheap appends | Unordered scans, extra seek per lookup |
| Hash-organized | By hash bucket | Hash itself is the index | Fast point lookups | Weak for range scans |
| Index-organized (IOT) | Key order, in the index | Index *is* the data | Saves a seek, cheap range scans | Secondary indexes must handle row relocation |

### Secondary-index referencing strategies

| Strategy | Read cost | Write / relocation cost | Where seen |
|----------|-----------|-------------------------|------------|
| Direct file offset | 1 seek (offset is final) | Every secondary index updates on every record relocation | Heap-file systems with few indexes or read-heavy workloads |
| Primary-key indirection | 2 lookups (secondary -> primary -> data) | Secondary indexes untouched by relocations | MySQL InnoDB |
| Hybrid (offset + PK) | Usually 1 seek; 2 on stale-offset fallback | Lazy repair on miss | Systems trading complexity for amortized best-case reads |

### Index classifications

| Classification | Definition |
|----------------|------------|
| Primary vs secondary | Primary = built on the primary key over the data file; secondary = anything else |
| Clustered vs non-clustered | Clustered = index order matches data file order; non-clustered = it doesn't |
| Unique vs non-unique | Primary indexes are unique per search key; secondary indexes may have many entries per key |

---

## 5. Buffering, Immutability, and Ordering

A storage engine is built on a data structure, but the engine's *personality* — its caching, recovery, and transactionality behaviour — comes from how it answers three orthogonal questions. Every structure in the rest of the book can be placed on these three axes, and most performance trade-offs in the book reduce to choosing a point in this cube.

**Buffering** asks whether the engine deliberately accumulates writes in memory before flushing. Some buffering is unavoidable (blocks are the minimum I/O unit), but *avoidable* buffering is a design choice. Lazy B-Trees add in-memory buffers to nodes to amortize I/O. Two-component **LSM Trees** buffer aggressively and combine buffering with immutability, which is how they turn random writes into sequential ones. **Immutability** asks whether a page can be modified in place or must be appended/written to a new location. Append-only designs (LSM Trees, Bitcask) and **copy-on-write** designs (Bw-Trees) never rewrite an old page, which makes recovery and concurrency simpler at the cost of needing compaction. **Ordering** asks whether records are stored in key order on disk. Key-ordered pages make range scans efficient but make random inserts expensive; insertion-ordered pages (Bitcask, WiscKey) reverse that trade-off.

These three choices are largely independent, which is why so many storage engines exist — the cube has eight corners and most are inhabited. The table below places the main examples on the axes.

```mermaid
graph LR
    subgraph Axes[Three orthogonal design axes]
        B[Buffering<br/>on / off]
        I[Immutability<br/>append-only / in-place]
        O[Ordering<br/>key-ordered / insertion-ordered]
    end
    Axes --> Space[Storage-engine design space]
    Space --> BT[B-Tree]
    Space --> LSM[LSM Tree]
    Space --> BW[Bw-Tree]
    Space --> BC[Bitcask]
    Space --> WK[WiscKey]
```

### Storage structures on the three axes

| Structure | Buffering | Mutability | Ordering | Optimised for |
|-----------|-----------|------------|----------|---------------|
| B-Tree (classic) | Minimal | In-place updates | Key-ordered | Balanced reads/writes, range scans |
| Lazy B-Tree | Per-node in-memory buffer | In-place (deferred) | Key-ordered | Write amplification reduction |
| LSM Tree | Heavy (memtable) | Append-only (SSTables) | Key-ordered (per level) | Write-heavy workloads |
| Bw-Tree | Delta records in memory | Copy-on-write (append-only) | Key-ordered | Concurrency, lock-free updates |
| Bitcask | None (hash in RAM) | Append-only log | Insertion order | Tiny-value writes, predictable latency |
| WiscKey | Value log buffering | Append-only value log | Keys ordered, values insertion-ordered | SSDs, avoiding write amplification on values |

### Cost of each axis

| Axis | Benefit | Cost |
|------|---------|------|
| Buffering | Amortized I/O, coalesced writes | Extra memory, crash-recovery complexity |
| Immutability | Cheap writes, simple concurrency | Compaction / GC overhead, space amplification |
| Ordering | Fast range scans | Expensive random inserts, page splits |

---

## Concrete Numbers: Latency Hierarchy and Columnar I/O Savings

The chapter leans on the fact that memory is "several orders of magnitude" faster than disk and that column stores avoid reading unrequested fields. The cell below makes both concrete using standard "latency numbers every programmer should know" values (referenced in the chapter via the Berkeley interactive-latency visualization) and a representative 100M-row, 10-column workload where a query touches only 2 columns.

In [ ]:
# Stdlib-only, Python 3.11+

# --- Latency hierarchy (typical "numbers every programmer should know") ---
# All values in nanoseconds.
latencies_ns = {
    "L1 cache reference":                 1,
    "L2 cache reference":                 4,
    "Main memory reference (DRAM)":     100,
    "SSD random read (4KB)":        150_000,     # ~0.15 ms
    "Round trip within datacenter": 500_000,     # ~0.5 ms
    "HDD seek":                  10_000_000,     # ~10 ms
}

print(f"{'Operation':<35}{'Latency':>18}{'vs DRAM':>14}")
print("-" * 67)
dram = latencies_ns["Main memory reference (DRAM)"]
for op, ns in latencies_ns.items():
    if ns < 1_000:
        human = f"{ns} ns"
    elif ns < 1_000_000:
        human = f"{ns/1_000:,.1f} us"
    else:
        human = f"{ns/1_000_000:,.2f} ms"
    ratio = ns / dram
    ratio_s = f"{ratio:.2f}x" if ratio >= 1 else f"1/{1/ratio:.0f}x"
    print(f"{op:<35}{human:>18}{ratio_s:>14}")

# Rule of thumb the chapter relies on:
ssd_vs_dram = latencies_ns["SSD random read (4KB)"] / dram
hdd_vs_dram = latencies_ns["HDD seek"] / dram
print(f"\nSSD random read is ~{ssd_vs_dram:,.0f}x slower than DRAM.")
print(f"HDD seek is      ~{hdd_vs_dram:,.0f}x slower than DRAM.")
print("This gap is *why* buffer managers and in-memory stores exist.")


# --- Row vs column I/O for a selective-projection query ---
# Workload: 100M user records, 10 fields, query selects 2 fields across ALL rows.
rows                 = 100_000_000
fields_per_row       = 10
bytes_per_field      = 16          # mix of ints, short strings, timestamps
fields_queried       = 2
page_size_bytes      = 8_192       # typical DB page

row_size_bytes       = fields_per_row * bytes_per_field
total_table_bytes    = rows * row_size_bytes

# Row-oriented: to read even one field, the entire row's page must be loaded.
# Scanning all rows -> scan the whole table.
row_oriented_bytes_read = total_table_bytes

# Column-oriented: only the queried columns' files are read.
column_oriented_bytes_read = rows * bytes_per_field * fields_queried

savings_ratio  = row_oriented_bytes_read / column_oriented_bytes_read
savings_pct    = 100 * (1 - column_oriented_bytes_read / row_oriented_bytes_read)

def gb(n): return n / (1024 ** 3)

print("\n--- 100M rows, 10 fields, query reads 2 fields ---")
print(f"Row size:                         {row_size_bytes} B")
print(f"Total table size:                 {gb(total_table_bytes):>8,.2f} GB")
print(f"Row-oriented  bytes read (scan):  {gb(row_oriented_bytes_read):>8,.2f} GB")
print(f"Column-oriented bytes read:       {gb(column_oriented_bytes_read):>8,.2f} GB")
print(f"I/O savings:                      {savings_ratio:.1f}x  ({savings_pct:.1f}% less)")

# What that I/O gap looks like in wall-clock time on an SSD
ssd_ns_per_page = latencies_ns["SSD random read (4KB)"]
pages_row  = row_oriented_bytes_read   // page_size_bytes
pages_col  = column_oriented_bytes_read // page_size_bytes
seconds_row = pages_row * ssd_ns_per_page / 1e9
seconds_col = pages_col * ssd_ns_per_page / 1e9
print(f"\nUpper-bound scan time on SSD (page-seek model):")
print(f"  Row-oriented:    ~{seconds_row:>8,.1f} s")
print(f"  Column-oriented: ~{seconds_col:>8,.1f} s")
print("Compression (which columnar does better) would widen the gap further.")

---

## Key Takeaways

1. **A DBMS is a pipeline of layers**: transport -> query processor -> optimizer -> execution engine -> storage engine. The storage engine itself decomposes into transaction, lock, access-method, buffer, and recovery managers.
2. **Where data lives is a first-order design decision.** In-memory engines and disk-based engines are not "same system, more RAM" — they use different data structures, different durability strategies, and different variable-size handling.
3. **Access patterns pick the layout.** Row-oriented is a good fit for OLTP-style whole-record access; column-oriented wins when queries scan many rows but only a few columns; wide-column stores are a separate family optimised for keyed access over families of related columns.
4. **Data files and index files are separate concerns.** Data can be heap-, hash-, or index-organized; indexes split into primary vs secondary and clustered vs non-clustered. The direct-offset vs primary-key indirection choice trades read latency against write amplification.
5. **Deletions are logical first, physical later.** Modern stores use tombstones and reclaim space during garbage collection rather than editing pages in place.
6. **Storage engines live in a 3D design space.** Buffering, immutability, and ordering are orthogonal axes — B-Trees, LSM Trees, Bitcask, Bw-Trees, and WiscKey differ mostly in which corner they occupy.
7. **Every later chapter is a zoom-in on one of these distinctions**, so having a precise vocabulary for them pays off repeatedly.

## See Also

- [[01-dbms-architecture]]
- [[02-memory-vs-disk-based-storage]]
- [[03-row-vs-column-oriented-layouts]]
- [[04-data-files-and-index-files]]
- [[05-buffering-immutability-ordering]]